# P13 — ReAct: sinergia entre razonar y actuar en modelos de lenguaje

## 1. Título y paper

**Paper:** *ReAct: Synergizing Reasoning and Acting in Language Models*  
**Autoría:** Shunyu Yao, Jeffrey Zhao, Dian Yu, Nan Du, Izhak Shafran, Karthik Narasimhan, Yuan Cao  
**Año y venue:** 2022 · arXiv:2210.03629 · ICLR 2023  
**Nivel:** L2 · **Motor:** `react`  
**Ficha completa:** [`P13_react`](../../papers/foundational/P13_react/README.md)

**Hito:** El modelo deja de ser solo un generador de texto y pasa a ser el controlador de un bucle que observa y actúa.

- [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El razonamiento en cadena (CoT) no consulta el mundo y alucina hechos; actuar sin razonar no descompone problemas de varios pasos.
2. Ejecutar una implementación mínima de la propuesta: Intercalar trazas de pensamiento y acciones sobre un entorno, de modo que cada observación real condicione el siguiente razonamiento.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10
- P11
- Wei et al. (2022), Chain-of-Thought


## 4. Intuición

Pensar en voz alta sin mirar nada lleva a inventar. Actuar sin pensar lleva a dar palos de ciego. ReAct alterna: pienso qué necesito, lo busco, leo lo que salió, y ese resultado real condiciona mi siguiente pensamiento.


## 5. Concepto mínimo

```text
bucle:  Thought_t → Action_t → Observation_t → Thought_{t+1} → …  → Finish
```

La observación viene del **entorno**, no del modelo. Ese es el anclaje que corrige la trayectoria: sin él, el razonamiento en cadena se aleja de los hechos sin darse cuenta.


## 6. Código explicado

El motor compara una estrategia solo-acción con el bucle completo sobre la misma pregunta compuesta.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('react', seed=7)['result']
print('pregunta:', r['pregunta'], '\n')
print('ACT-ONLY →', r['act_only']['answer'], '· pasos:', r['pasos']['act_only'])
print('REACT    →', r['react']['answer'], '· pasos:', r['pasos']['react'], '\n')
for paso in r['react']['trace']:
    print('  💭', paso['thought'])
    print('  🔧', paso['act'], '→', paso['obs'])

## 7. Predicción antes de ejecutar

1. ¿Por qué falla la estrategia solo-acción en una pregunta de dos saltos?
2. ¿Cuántas llamadas a la herramienta necesita ReAct como mínimo?
3. Si la primera observación fuera errónea, ¿el bucle lo detectaría?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
KB = {'capital de francia': 'Paris', 'poblacion de paris': '2 100 000'}

def buscar(clave):
    return KB.get(clave.lower(), 'sin resultados')

def bucle(pregunta, max_pasos=4):
    traza, respuesta = [], None
    consulta = 'capital de francia'
    for paso in range(max_pasos):
        obs = buscar(consulta)
        traza.append({'paso': paso, 'accion': f'buscar({consulta})', 'obs': obs})
        if obs == 'sin resultados':
            traza.append({'paso': paso, 'decision': 'PARAR: la herramienta no sabe'})
            break
        if consulta.startswith('poblacion'):
            respuesta = obs
            break
        consulta = f'poblacion de {obs.lower()}'
    return {'respuesta': respuesta, 'traza': traza}

show(bucle('cuantos habitantes tiene la capital de francia'))

## 9. Salida interpretable

La traza muestra cómo la observación `Paris` **construye** la consulta siguiente. Sin ese encadenamiento la pregunta es irresoluble con una sola búsqueda, por muy bueno que sea el modelo.


## 10. Comentario pedagógico

Una traza legible no garantiza fidelidad: el texto del «pensamiento» es una generación más y puede no describir el proceso real. Sirve para depurar y auditar decisiones, no como prueba de cómo razonó el modelo.


## 11. Error o anti-patrón deliberado

Anti-patrón: bucle sin criterio de parada. Si la herramienta falla, el agente reintenta para siempre y quema presupuesto.


In [ ]:
intentos = 0
for _ in range(50):                      # simulación acotada de un bucle infinito
    intentos += 1
    obs = buscar('dato que no existe')
    if obs != 'sin resultados':
        break
print(f'la herramienta devolvió «sin resultados» {intentos} veces y el agente siguió intentando')

## 12. Corrección

Corrección: límite de pasos, detección de repetición y escalamiento explícito.


In [ ]:
def bucle_seguro(consulta, max_pasos=3):
    vistas = set()
    for paso in range(max_pasos):
        if consulta in vistas:
            return {'estado': 'ABORTADO', 'motivo': 'consulta repetida', 'pasos': paso}
        vistas.add(consulta)
        obs = buscar(consulta)
        if obs == 'sin resultados':
            return {'estado': 'ESCALADO', 'motivo': 'herramienta sin datos', 'pasos': paso + 1}
        return {'estado': 'OK', 'obs': obs, 'pasos': paso + 1}
    return {'estado': 'ABORTADO', 'motivo': 'límite de pasos', 'pasos': max_pasos}

show(bucle_seguro('dato que no existe'))
show(bucle_seguro('capital de francia'))

## 13. Desafío guiado

Haz que la base de conocimiento devuelva un dato erróneo y comprueba que el bucle lo propaga sin dudar.


In [ ]:
KB['capital de francia'] = 'Berlin'          # dato corrupto
resultado = bucle('cuantos habitantes tiene la capital de francia')
show(resultado)
print('→ el agente no cuestiona la observación: la fiabilidad de la herramienta es su techo')
KB['capital de francia'] = 'Paris'           # restaurar

## 14. Desafío autónomo

Implementa ReAct sobre una API pública y gratuita. Añade un verificador que compruebe cada observación contra una segunda fuente. Mide cuántas respuestas cambian al añadir el verificador y cuánto cuesta en llamadas.


## 15. Evidencia de aprendizaje

Guarda la traza completa, el caso de bucle sin parada, la versión con criterio de parada y el experimento del dato corrupto.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P13_react/README.md) · evaluación formal: [`assessments/papers/P13_react.md`](../../assessments/papers/P13_react.md)


## 16. Cierre

El modelo ya controla un bucle. Falta que aprenda **cuándo** conviene llamar a una herramienta, en lugar de que se lo digamos nosotros.


## 17. Conexión con el siguiente hito

- P14
- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
